# 2.1 User activity

**Question 1:** how many events does each user have?

Why it matters: a recommender can only learn from users who have enough history.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from realtime_recsys.config import PROCESSED_DATA_DIR

events = pl.read_parquet(PROCESSED_DATA_DIR / "events_clean.parquet")

## Events per user

Group by user and count rows. The result has one row per user.

In [ ]:
per_user = events.group_by("user_id").agg(pl.len().alias("n_events"))
n_users = per_user.height
print(f"users: {n_users:,}")
per_user["n_events"].describe()

## Share of users with few events

For each threshold k: what fraction of users has k events or fewer?

In [ ]:
for k in [1, 2, 5, 10, 50]:
    share = (per_user["n_events"] <= k).sum() / n_users
    print(f"<= {k:>2} events: {share:6.1%} of users")

## Distribution

Both axes use a log scale because a few users have thousands of events.

In [ ]:
counts = per_user["n_events"].to_numpy()
bins = np.logspace(0, np.log10(counts.max()), 60)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(counts, bins=bins)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("events per user (log scale)")
ax.set_ylabel("number of users (log scale)")
ax.set_title("Events per user, October 2019")
plt.show()